1. Installatie (%%capture !pip install unsloth...)

In [1]:
%%capture
!pip install unsloth
!pip install --upgrade trl transformers accelerate peft datasets

2. GPU check (torch.cuda.is_available())


In [2]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

3. Model laden (FastLanguageModel.from_pretrained)


In [3]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-3-1b-it",
    max_seq_length = 2048,
    load_in_4bit = True,
)

print("Model geladen!")

4. LoRA configureren (get_peft_model)


In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

print("LoRA geconfigureerd!")

5. Dataset laden (GitHub URL cel)


In [5]:
import json
import urllib.request
from datasets import Dataset

url = "https://raw.githubusercontent.com/ammarmangala/geopolio-finetune/main/data/geopolio_dataset_2099s_global_multidecade.json"

with urllib.request.urlopen(url) as response:
    raw_data = json.load(response)

def format_sample(sample):
    return {
        "text": f"""### Instruction:
{sample['instruction']}

### Input:
{sample['input']}

### Response:
{sample['output']}"""
    }

formatted = [format_sample(s) for s in raw_data]
dataset = Dataset.from_list(formatted)

print(f"Dataset geladen: {len(dataset)} samples")
print("\nVoorbeeld:")
print(dataset[0]["text"][:300])

6. Trainer en training


In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    args = SFTConfig(
        output_dir = "./output",
        num_train_epochs = 5,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        learning_rate = 2e-4,
        warmup_steps = 80,
        lr_scheduler_type = "cosine",
        logging_steps = 50,
        save_strategy = "no",
        fp16 = False,
        optim = "adamw_8bit",
        seed = 42,
        dataset_text_field = "text",
        packing = False,
        padding_free = False,
    ),
)

print("Trainer klaar, training start...")
trainer.train()

7. Testen


In [ ]:
FastLanguageModel.for_inference(model)

prompt = """### Instruction:
Analyze the geopolitical risk of the following situation for European retail investors.

### Input:
The United States has announced sweeping tariffs on all European steel and aluminum imports, triggering EU retaliation threats.

### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens = 300,
    temperature = 0.7,
    do_sample = True,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response[len(prompt):])

8. Opslaan en uploaden naar Hugging Face


In [ ]:
from huggingface_hub import login

login(token="HF_TOKEN")

slaat op lokaal op de Colab runtime. Die bestanden verdwijnen als de sessie stopt. Je wilt push_to_hub_gguf zodat het direct naar Hugging Face gaat.

In [ ]:
model.save_pretrained_gguf(
    "ammarmangala/geopolio-risk-global-200s-gemma3-1b",
    tokenizer,
    quantization_method = "q4_k_m",
    token = "HF_TOKEN",
)

In [ ]:
from huggingface_hub import login
login(token="HF_TOKEN")

model.push_to_hub_gguf(
    "ammarmangala/geopolio-risk-global-multidecade-2099s-gemma3-1b",
    tokenizer,
    quantization_method = "q4_k_m",
    token = "HF_TOKEN",
)